<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: From Single Agent to Multi-Agent Workflow </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Demonstrate the transition from a **single agent** to a **multi-agent workflow**.  
This notebook shows why single agents have limitations and how splitting responsibilities into specialized agents with basic coordination improves performance and clarity.

**Key Topics Covered Today:**
- Limitations of single-agent systems
- Benefits of role specialization
- Basic task delegation pattern
- Simple inter-agent communication
- From monolithic agent → Multi-agent team

---

## 1. Setup

In [4]:
!pip install -q --force-reinstall "langchain>=0.1.0,<1.0" "langchain-core==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" langsmith duckduckgo-search ddgs faiss-cpu beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
langgraph 1.2.11 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
google-adk 2.7.0 requires websockets<16,>=15.0.1, but you have websockets 17.0.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.2 which is incompatible.
google-genai 2.12.1 requires websockets<17.0,>=13.0.0, but you have websockets 17.0.1 which is incompatible.
la

In [1]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete!")

Enter your OpenAI API key: ··········
Setup complete!


## 2. Single Agent Workflow (Review from Week 3)

Let's start with a single agent that handles the entire travel budget planning task.

In [2]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langsmith import Client

search_tool = DuckDuckGoSearchRun(name="Web_Search")

@tool
def calculate_math(expression: str) -> str:
    """Perform math calculations (addition, multiplication, percentages, etc.)"""
    try:
        allowed = {"__builtins__": {}}
        return str(eval(expression, allowed, {}))
    except Exception as e:
        return f"Error: {str(e)}"

tools = [search_tool, calculate_math]
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

single_agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

single_executor = AgentExecutor(
    agent=single_agent,
    tools=tools,
    verbose=True,
    max_iterations=10
)

print("Single Agent Ready")

Single Agent Ready


## 3. Demo: Single Agent (Monolithic Approach)

In [3]:
single_query = """Plan a budget trip from Riyadh to Istanbul.
Find approximate cheapest round-trip flight price in USD,
average hotel price per night in USD, current USD to SAR exchange rate,
then calculate total budget in SAR for flights + 7 nights hotel,
and finally calculate what percentage the flight cost is of the total budget."""

response_single = single_executor.invoke({"input": single_query})

print("\n" + "="*70)
print("SINGLE AGENT FINAL ANSWER:")
print(response_single["output"])



> Entering new AgentExecutor chain...
I need to gather information on the following: the approximate cheapest round-trip flight price from Riyadh to Istanbul, the average hotel price per night in Istanbul, the current USD to SAR exchange rate, and then perform calculations based on this data to find the total budget in SAR for flights and 7 nights of hotel stay, as well as the percentage of the flight cost relative to the total budget.

Action: Web_Search  
Action Input: "cheapest round-trip flight price from Riyadh to Istanbul"  September is the cheapest month to fly from Riyadh (RUH) to Istanbul (IST), with average round-trip fares around $228. This price is about 48% lower than the average throughout the year, likely due to reduced travel demand after the summer peak. King Khalid International Airport in Riyadh (source: photo by Kskhh on Wikipedia) The sole available mode of transportation from Riyadh to Istanbul is by air travel. Flights from Riyadh to Istanbul are incredibly pop

## 4. Why Move to Multi-Agent Systems?

**Limitations of Single Agent:**
- Does everything → becomes slow and unfocused
- Hard to debug when something goes wrong
- One agent must be expert in many domains
- Poor separation of concerns
- Harder to scale or add new capabilities

**Benefits of Multi-Agent:**
- Role specialization (Researcher, Analyst, Calculator, Supervisor)
- Better reasoning per role
- Easier to trace and debug
- Natural task delegation
- More modular and maintainable

---

## 5. Building a Simple Multi-Agent Workflow

### Creating Specialized Agents

We will create two specialized agents:
1. **Research Agent** — Good at searching and gathering information
2. **Math Agent** — Good at performing calculations and percentages

In [ ]:
from langchain.agents import create_react_agent, AgentExecutor
from langsmith import Client
from langchain_openai import ChatOpenAI # Import ChatOpenAI if not already imported

client = Client()

# Research Agent (focuses only on searching)
research_prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)
research_agent = create_react_agent(llm=llm, tools=[search_tool], prompt=research_prompt)

research_executor = AgentExecutor(
    agent=research_agent,
    tools=[search_tool],
    verbose=False,   # Set to True if you want to see details
    max_iterations=15 # Increased max_iterations
)

# Math Agent (focuses only on calculations)
math_prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)
# Create a new LLM instance specifically for the math agent with a stop sequence
math_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, stop=["\nObservation:"])
math_agent = create_react_agent(llm=math_llm, tools=[calculate_math], prompt=math_prompt)

math_executor = AgentExecutor(
    agent=math_agent,
    tools=[calculate_math],
    verbose=False,
    max_iterations=15, # Increased max_iterations
    handle_parsing_errors=True # Added this line to handle parsing errors
)

print("Research Agent and Math Agent created")

## 6. Basic Task Delegation Pattern

A simple **Supervisor → Worker** pattern:
- Supervisor decides what to do
- Delegates research to Research Agent
- Delegates calculations to Math Agent
- Combines results

In [13]:
import re

def multi_agent_travel_planner(query: str):
    print("Supervisor: Breaking down the task...\n")

    # Step 1: Research Agent gathers data
    print("Delegating to Research Agent...")
    research_result = research_executor.invoke({
        "input": "Find current cheapest round-trip flight price Riyadh to Istanbul in USD, "
                 "average hotel price per night in Istanbul in USD, "
                 "and current USD to SAR exchange rate."
    })

    research_data = research_result["output"]
    print("Research complete.\n")

    # Step 2: Supervisor parses research data for math agent
    print("Supervisor: Parsing research data for Math Agent...")

    flight_price_match = re.search(r"flight price.*?: \$(\d+)", research_data)
    flight_price = int(flight_price_match.group(1)) if flight_price_match else 0

    # For average hotel price, taking the lower bound of the range for 'cheapest trip'
    hotel_price_match = re.search(r"hotel price per night.*?: \$(\d+)", research_data)
    hotel_price_per_night = int(hotel_price_match.group(1)) if hotel_price_match else 0

    exchange_rate_match = re.search(r"exchange rate.*?: (\d+\.\d+)", research_data)
    exchange_rate = float(exchange_rate_match.group(1)) if exchange_rate_match else 0.0

    print(f"Parsed data: Flight=${flight_price}, Hotel/night=${hotel_price_per_night}, Rate={exchange_rate}\n")

    # Step 3: Math Agent performs calculations with structured input
    print("Delegating to Math Agent for calculations...")
    math_input = (
        f"Given the following data:\n"
        f"Flight price (USD): {flight_price}\n"
        f"Hotel price per night (USD): {hotel_price_per_night}\n"
        f"Number of nights: 7\n"
        f"USD to SAR exchange rate: {exchange_rate}\n\n"
        f"Please calculate:\n"
        f"1. Total budget in SAR for the flight + 7 nights hotel.\n"
        f"2. What percentage the flight cost is of the total budget.\n"
        f"Provide only the final numerical answers for each calculation, clearly labeled."
    )

    math_result = math_executor.invoke({"input": math_input})

    print("Calculations complete.\n")

    # Final Answer
    final_answer = f"""MULTI-AGENT RESULT

Research Findings:
{research_data}

Calculation Results:
{math_result["output"]}

 Task completed through coordinated agents."""

    return final_answer

# Run the multi-agent workflow
multi_result = multi_agent_travel_planner("Plan budget trip from Riyadh to Istanbul")

print("="*70)
print("MULTI-AGENT FINAL OUTPUT:")
print(multi_result)

Supervisor: Breaking down the task...

Delegating to Research Agent...
Research complete.

Supervisor: Parsing research data for Math Agent...
Parsed data: Flight=$215, Hotel/night=$49, Rate=0.0

Delegating to Math Agent for calculations...
Calculations complete.

MULTI-AGENT FINAL OUTPUT:
MULTI-AGENT RESULT

Research Findings:
- Current cheapest round-trip flight price from Riyadh to Istanbul: $215
- Average hotel price per night in Istanbul: $49
- Current USD to SAR exchange rate: Unable to retrieve the specific rate.

Calculation Results:
1. Total budget in SAR: 2092.5
2. Flight cost percentage of total budget: 38.53%

 Task completed through coordinated agents.


## 7. Reflection Questions

1. What were the main limitations you observed in the single agent?
2. How did role specialization help in the multi-agent version?
3. What kind of communication happened between agents?
4. In what real-world scenarios would multi-agent systems be more beneficial?
5. What challenges might arise when scaling to 4–5 specialized agents?
